In [0]:
# Databricks notebook source
from typing import Any

import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    array_join,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    posexplode,
    regexp_replace,
    sha2,
    split,
    trim,
    when,
    filter as array_filter,
    transform as array_transform,
)

dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_auction_real_estate_extfile(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :

        date_filter = f"businessdate = {run_date!r}"

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.AuctionRealEstate"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "extfile",
                )        
        else:
            logging.info("Tables doe not exist, no need to delete entries for business date!")


        # ================== auctions ==================

        auctions_extra_filters = None

        if auctions_extra_filters is not None:
            auctions_extra_filters.append(date_filter)
            auctions_final_filters = " AND ".join(auctions_extra_filters)
        else:
            auctions_final_filters = date_filter

        auctions_selected_fields = [
            "UniqueCode",
            "Source",
            "AuctionObJID",
            "AuctionID",
            "DebtorVatNumber",
            "DebtorDescr",
            "AuctionDate",
            "PublishedDate",
            "eAuctionPostedDate",
            "FirstOfferPrice",
            "SeizureReportNumber",
            "SeizureReportYear",
            "HastenerDescr",
            "HastenerVAT",
            "FinalPrice",
            "auctionstatusDescr",
            "AuctioneerName",
            "AuctioneerEmail",
            "AuctioneerPhone",
            "TitleGR",
            "Area",
            "District",
            "Municipality",
            "Prefecture",
            "GeoCoordinatesX",
            "GeoCoordinatesY",
            "GeoLevel",
            "Address",
            "PostCode",
            "CategoryGR",
            "SubCategoryGR",
            "LandeaUrl",
            "FloorRawValue",
            "ConstructionYear",
            "SurfaceSqM",
            "Tags",
            "TagValues",
            "Active",
            "TNMergeId",
            "GroupObjID",
            "KAEK",
            "file_date",
            "file_name",
            "businessdate",
        ]

        auctions_df = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "newsphone_auctions",
            auctions_final_filters,
            auctions_selected_fields,
        )


        # ================== rcg_mbp_snapshot ==================

        rcg_extra_filters = None

        if rcg_extra_filters is not None:
            rcg_extra_filters.append(date_filter)
            rcg_final_filters = " AND ".join(rcg_extra_filters)
        else:
            rcg_final_filters =  f"""
            businessdate = (
                SELECT MAX(businessdate)
                FROM {catalog_landed}.rcg_mbp_snapshot
                WHERE businessdate <= '{run_date}'
            )
            """

        rcg_selected_fields = [
            "rcg_provider_auctionobjid",
            "rcg_provider_mbp_price",
            "rcg_provider_uw_price",
            "rcg_provider_acquired_by",
            "rcg_provider_file_date",
            "rcg_provider_eligibility",
            "rcg_provider_initiator",
            "rcg_provider_Servicer",
            "rcg_provider_Auction_result_updated",
            "rcg_provider_suspension_reason_grouped",
            "rcg_provider_suspension_reason_full",
            "rcg_provider_eligibility_flag_property",
            "rcg_provider_eligible_for_reschedule",
            "rcg_provider_marketability_score",
            "rcg_provider_auction_round",
        ]

        rcg_df = load_landed_delta_table(
            spark,
            catalog_landed,
            "rcg_mbp_snapshot",
            rcg_final_filters,
            rcg_selected_fields,
        )


        # ================== Join ==================

        final_df = auctions_df.join(
            rcg_df,
            auctions_df.AuctionObJID == rcg_df.rcg_provider_auctionobjid,
            "left",
        )

        # HashKeys
        final_df_AuctionRealEstateId_primary_key_fields = [
            final_df.AuctionObJID,
            lit("extfile"),
        ]

        final_df = final_df.withColumn(
            "AuctionRealEstateId",
            sha2(
                concat_ws(";", *final_df_AuctionRealEstateId_primary_key_fields),
                256,
            ),
        )       


        # clean VAT, keep only valid 9-digit VAT numbers separated by '||'.
        vat_tokens = split(col("DebtorVatNumber"), "[|][|]")

        # strip leading/trailing non digit characters
        vat_stripped = array_transform(
            vat_tokens,
            lambda t: regexp_replace(t, "^[^0-9]+|[^0-9]+$", ""),
        )
        
        # filter valid VAT numbers
        vat_valid = array_filter(
            vat_stripped, lambda t: t.rlike("^[0-9]{9}$")
        )
        
        # rebuild column
        vat_clean = array_join(vat_valid, "||")
        
        final_df = final_df.withColumn(
            "DebtorVatNumber",
            when(vat_clean == lit(""), lit(None)).otherwise(vat_clean),
        )

        # ================== Selection of Final Columns ==================

        result_df_extfile = final_df.select(
            final_df.AuctionRealEstateId.alias("AuctionRealEstateId"),
            final_df.Source.alias("OriginSourceSystem"),
            lit("extfile").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            final_df.AuctionObJID.cast("string").alias("AuctionRealEstateKey"),
            final_df.UniqueCode.alias("AuctionRealEstateCode"),
            final_df.AuctionID.alias("AuctionCode"),
            final_df.AuctionObJID.cast("string").alias("AuctionObjKey"),
            final_df.GroupObjID.alias("RealEstateGroupCode"),
            lit(None).alias("PartyKey"),
            lit(None).alias("RealEstateKey"),
            lit(None).alias("RealEstateOwnershipKey"),
            lit(None).alias("CaseCode"),
            lit(None).alias("DebtorPartyKey"),
            final_df.DebtorVatNumber.alias("OriginDebtorTaxRegistrationNumber"),
            final_df.DebtorDescr.alias("OriginDebtorDescription"),
            final_df.AuctionDate.cast("Date").alias("AuctionDate"),
            final_df.FinalPrice.cast("String").alias("AuctionFinalPrice"),
            final_df.PublishedDate.alias("AuctionPublishedDate"),
            final_df.eAuctionPostedDate.alias("AuctionPostedDate"),
            final_df.FirstOfferPrice.cast("String").alias("AuctionFirstOfferPrice"),
            final_df.SeizureReportNumber.alias("SeizureReportNumber"),
            final_df.SeizureReportYear.alias("SeizureReportYear"),
            lit(None).alias("HastenerCode"),
            final_df.HastenerDescr.alias("Hastener"),
            final_df.HastenerVAT.alias("HastenerTaxRegistrationNumber"),
            final_df.auctionstatusDescr.alias("AuctionStatusCode"),
            final_df.auctionstatusDescr.alias("AuctionStatus"),
            final_df.AuctioneerName.alias("AuctioneerName"),
            final_df.AuctioneerEmail.alias("AuctioneerEmail"),
            final_df.AuctioneerPhone.alias("AuctioneerPhone"),
            final_df.TitleGR.alias("TitleGR"),
            final_df.Area.alias("Area"),
            final_df.District.alias("District"),
            final_df.Municipality.alias("Municipality"),
            final_df.Prefecture.alias("Prefecture"),
            final_df.GeoCoordinatesX.alias("GeoCoordinatesX"),
            final_df.GeoCoordinatesY.alias("GeoCoordinatesY"),
            final_df.GeoLevel.alias("GeoLevel"),
            final_df.Address.alias("Address"),
            final_df.PostCode.alias("PostalCode"),
            final_df.CategoryGR.alias("Category"),
            final_df.SubCategoryGR.alias("SubCategory"),
            final_df.LandeaUrl.alias("LandeaUrl"),
            final_df.FloorRawValue.alias("FloorRawValue"),
            final_df.ConstructionYear.alias("ConstructionYear"),
            final_df.SurfaceSqM.alias("SurfaceSqm"),
            final_df.Tags.alias("Tags"),
            final_df.TagValues.alias("TagValues"),
            final_df.Active.cast("boolean").alias("ActiveFlag"),
            final_df.TNMergeId.cast("String").alias("TNMergeCode"),
            final_df.KAEK.alias("LandRegistration"),
            lit(None).alias("BidderCode"),
            lit(None).alias("DateEntered"),
            lit(None).alias("EnteredBy"),
            lit(None).alias("AuctionOwnershipTypeCode"),
            lit(None).alias("AuctionOwnershipType"),
            lit(None).alias("FirstBidder"),
            lit(None).alias("LawFirm"),
            final_df.file_date.alias("FileDate"),
            final_df.file_name.alias("FileName"),
            final_df.rcg_provider_auctionobjid,
            final_df.rcg_provider_mbp_price,
            final_df.rcg_provider_uw_price,
            final_df.rcg_provider_acquired_by,
            final_df.rcg_provider_file_date,
            final_df.rcg_provider_eligibility,
            final_df.rcg_provider_initiator,
            final_df.rcg_provider_Servicer,
            final_df.rcg_provider_Auction_result_updated,
            final_df.rcg_provider_suspension_reason_grouped,
            final_df.rcg_provider_suspension_reason_full,
            final_df.rcg_provider_eligibility_flag_property,
            final_df.rcg_provider_eligible_for_reschedule,
            final_df.rcg_provider_marketability_score,
            final_df.rcg_provider_auction_round,
            final_df.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df_extfile.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")


    logging.info("We did it!")

In [0]:
load_curated_auction_real_estate_extfile(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)